# Ising Trace Diagnostics

Run `just example ising_1d` from the repository root first. The example writes `target/ising_1d_trace.csv`, which this notebook reads with Polars for trace inspection and acceptance statistics.

In Colab or another external notebook runtime, install `polars` and `matplotlib` if needed, then upload or mount the generated CSV.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

root_trace = Path("target/ising_1d_trace.csv")
notebook_trace = Path("../target/ising_1d_trace.csv")
trace_path = root_trace if root_trace.exists() else notebook_trace
if not trace_path.exists():
    raise FileNotFoundError(
        "Could not find Ising trace CSV. Checked "
        f"{root_trace} and {notebook_trace}. Run `just example ising_1d` first."
    )
trace = pl.read_csv(trace_path)
trace.head()

In [ ]:
summary = trace.group_by("chain_id").agg(
    pl.len().alias("steps"),
    pl.col("accepted").sum().alias("accepted"),
    pl.col("proposed").sum().alias("proposed"),
    pl.col("energy").mean().alias("mean_energy"),
    pl.col("magnetization").mean().alias("mean_magnetization"),
).with_columns(
    pl.when(pl.col("proposed") > 0)
    .then(pl.col("accepted") / pl.col("proposed"))
    .otherwise(0)
    .alias("acceptance_rate"),
)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(trace["step"].to_list(), trace["energy"].to_list())
ax.set_title("Energy trace")
ax.set_xlabel("step")
ax.set_ylabel("energy")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(trace["step"].to_list(), trace["magnetization"].to_list())
ax.set_title("Magnetization trace")
ax.set_xlabel("step")
ax.set_ylabel("magnetization per spin")

The exported columns are intentionally plain: `chain_id`, `step`, `accepted`, `proposed`, `log_prob`, then one column per observable. Later autocorrelation, integrated autocorrelation time, ESS, ESS-rate, and R-hat diagnostics can consume the same table without depending on plotting code.